# SPORTLIS U-Net — Extended Domain + Extended Period (1985–2024)

Estensione di `sportlis_unet_NARR_TM_lean_loyo_TPI_canopy.ipynb` su:
- **dominio geografico esteso**: lat [35.005, 49.495], lon [−124.93, −104.01] → griglia 484×698 @ 3 km
- **periodo esteso**: 1985–2024 (NARR disponibile dal 1979)

Il notebook è strutturato in blocchi eseguibili in sequenza:
1. Scarica NARR (solo `air.2m` e `apcp`) per il nuovo bbox
2. Calcola pesi di regridding NARR→SPORTLIS (una-tantum)
3. Estrae static features dalle nc4 SPORTLIS estese
4. Costruisce zarr per anno (precip, tair, swe_target_filled, swe_mask, is_snowy_time)
5. Training U-Net LOYO
6. XAI permutation importance

**Canali input (16, stessa architettura del lean+TPI+canopy baseline):**
```
7 TM   : precip_7d, precip_30d, precip_60d, precip_wytd, tair_30d_mean, tair_30d_max, degree_day_30d
2 coord: lat_norm, lon_norm
7 topo : elevation, slope, aspect_sin, aspect_cos, tpi_short, tpi_long, canopy_fraction
```

In [ ]:
# ── 0. IMPORTS ────────────────────────────────────────────────────────────────
import os, gc, time, json, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter1d, uniform_filter
from scipy.spatial import cKDTree

import netCDF4 as nc
import zarr

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
print('zarr version:', zarr.__version__)

In [ ]:
# ── 1. PATHS & CONFIGURATION ──────────────────────────────────────────────────
import os, socket
from pathlib import Path

# ── Auto-detect environment ───────────────────────────────────────────────────
_ON_NCAR  = Path('/glade/u/home/cionni').exists()
_ON_SMCE  = (not _ON_NCAR) and Path('/home/jovyan/efs').exists()
_ON_MAC   = not (_ON_NCAR or _ON_SMCE)

if _ON_NCAR:
    NCAR_HOME         = Path('/glade/u/home/cionni')
    NCAR_SCRATCH      = NCAR_HOME / 'derecho_scratch'
    PROJECT_DIR       = NCAR_HOME / 'unet_sportlis'
    SPORTLIS_DATA_DIR = NCAR_SCRATCH / 'sportlis_swe'       # sportlis_swe_*.nc4
    NARR_RAW_DIR      = NCAR_SCRATCH / 'narr_extended_raw'
    ZARR_DIR          = NCAR_SCRATCH / 'zarr_extended'
    MEMMAP_DIR        = NCAR_SCRATCH / 'memmap_extended'
    OUTPUT_DIR        = PROJECT_DIR  / 'output_extended'
    AUX_DIR           = PROJECT_DIR  / 'auxiliary'
elif _ON_SMCE:
    EFS_ROOT          = Path('/home/jovyan/efs/icionni')
    SPORTLIS_DATA_DIR = EFS_ROOT / 'sportlis_swe'
    NARR_RAW_DIR      = EFS_ROOT / 'narr_extended_raw'
    ZARR_DIR          = EFS_ROOT / 'zarr_extended'
    MEMMAP_DIR        = EFS_ROOT / 'memmap_extended'
    OUTPUT_DIR        = EFS_ROOT / 'output_extended'
    AUX_DIR           = EFS_ROOT / 'auxiliary'
else:  # Mac
    PROJECT_ROOT      = Path('/Users/irene/PROJECTS.index/Hydrology/projects/sportlis_project')
    SPORTLIS_DATA_DIR = PROJECT_ROOT / 'data/inputs'
    NARR_RAW_DIR      = PROJECT_ROOT / 'narr_extended_raw'
    ZARR_DIR          = PROJECT_ROOT / 'prepared_pp_narr_sportlis_extended'
    MEMMAP_DIR        = PROJECT_ROOT / 'memmap_extended'
    OUTPUT_DIR        = Path('sportlis_unet_extended_loyo_outputs')
    AUX_DIR           = PROJECT_ROOT / 'auxiliary'

STATIC_FILE = AUX_DIR / 'sportlis_static_extended.nc'
CANOPY_FILE = AUX_DIR / 'sportlis_canopy_extended_3km.nc'

for d in [NARR_RAW_DIR, ZARR_DIR, AUX_DIR, OUTPUT_DIR, MEMMAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_env = 'NCAR Casper/Derecho' if _ON_NCAR else ('SMCE' if _ON_SMCE else 'Mac locale')
print(f'Ambiente  : {_env}')
print(f'SWE data  : {SPORTLIS_DATA_DIR}')
print(f'ZARR      : {ZARR_DIR}')
print(f'MEMMAP    : {MEMMAP_DIR}')
print(f'OUTPUT    : {OUTPUT_DIR}')

# ── Dominio SPORTLIS esteso ───────────────────────────────────────────────────
EXT_LAT_MIN, EXT_LAT_MAX = 35.005, 49.495
EXT_LON_MIN, EXT_LON_MAX = -124.93, -104.01
EXT_H, EXT_W = 484, 698

# ── NARR download ─────────────────────────────────────────────────────────────
NARR_BASE = 'https://psl.noaa.gov/thredds/dodsC/Datasets/NARR/monolevel'
NARR_BBOX = dict(lat_min=34.5, lat_max=50.0, lon_min=-126.0, lon_max=-103.5)
NARR_VARS_NEEDED = {'air.2m': 'tair', 'apcp': 'precip'}
NARR_OPEN_KW = dict(decode_times=True, chunks={'time': 240})

# ── Rolling windows (NARR 3-h) ────────────────────────────────────────────────
DT_HOURS        = 3
STEPS_PER_DAY   = 8
WINDOW_7D       = 7  * STEPS_PER_DAY
WINDOW_30D      = 30 * STEPS_PER_DAY
WINDOW_60D      = 60 * STEPS_PER_DAY
TEMPORAL_BUFFER = 60 * STEPS_PER_DAY

# ── Canali ────────────────────────────────────────────────────────────────────
LEAN_TEMPORAL_VARS = ['precip_7d','precip_30d','precip_60d','precip_wytd',
                      'tair_30d_mean','tair_30d_max','degree_day_30d']
INPUT_VARS = list(LEAN_TEMPORAL_VARS)
TOPO_VARS  = ['elevation','slope','aspect_sin','aspect_cos',
              'tpi_short','tpi_long','canopy_fraction']
TPI_RADII  = {'tpi_short': 5, 'tpi_long': 10}
TARGET_VAR = 'swe_target_filled'
MASK_VAR   = 'swe_mask'

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE     = 8
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
EPOCHS         = 30
PATIENCE       = 6
DROPOUT_P      = 0.1
PATCH_SIZE     = 128
USE_SNOWY_ONLY = True
AUGMENT_NOISE  = True
ADD_DOY        = False
NUM_WORKERS    = 4 if (_ON_NCAR or _ON_SMCE) else 0

LOYO_STEP = 5

LOSS_LOG_BETA     = 0.2
LOSS_MM_BETA      = 20.0
LOSS_MM_WEIGHT    = 0.3
LOSS_TARGET_ALPHA = 2.0
LOSS_BIAS_WEIGHT  = 0.05

# ── Efficienza memoria ────────────────────────────────────────────────────────
REGRID_CHUNK        = 60
BUFFER_DAYS_CUTOFF  = 75
USE_MEMMAP          = True
N_PATCHES_PER_EPOCH = 3000
TIME_STRIDE_TRAIN   = 2

CHECKPOINT_TEMPLATE = 'best_unet_extended_LOYO_test{test}.pt'
FOLD_METRICS_CSV    = OUTPUT_DIR / 'loyo_extended_fold_metrics.csv'


## Sezione 1 — Scopri i file SPORTLIS estesi e i loro anni

In [ ]:
# ── 2. SCOPRI FILE SPORTLIS SWE ───────────────────────────────────────────────
import glob

nc4_files = sorted(glob.glob(str(SPORTLIS_DATA_DIR / 'sportlis_swe_*.nc4')))
print(f'File nc4 trovati: {len(nc4_files)}')
for f in nc4_files:
    print(' ', f)

def parse_sportlis_timestamps(sp_ds):
    """Legge timestamps dal file nc4 SPORTLIS (formato YYYYMMDDHH)."""
    raw = sp_ds.variables['timestamps'][:]
    as_str = [str(s) for s in raw]
    return pd.to_datetime(as_str, format='%Y%m%d%H')

# Mappa file → anni coperti
file_year_map = {}   # {year: filepath}
for f in nc4_files:
    sp = nc.Dataset(f)
    try:
        ts = parse_sportlis_timestamps(sp)
        years_in_file = sorted(set(ts.year))
        print(f'  {Path(f).name}: {ts[0]} → {ts[-1]}  ({len(ts)} h, anni: {years_in_file})')
        for y in years_in_file:
            if y not in file_year_map:
                file_year_map[y] = f
    finally:
        sp.close()

YEARS_AVAILABLE = sorted(file_year_map.keys())
print(f'\nAnni SPORTLIS disponibili ({len(YEARS_AVAILABLE)}): {YEARS_AVAILABLE[0]}–{YEARS_AVAILABLE[-1]}')

## Sezione 2 — Download NARR (air.2m + apcp) per dominio esteso

In [ ]:
# ── 3. FUNZIONI NARR DOWNLOAD ─────────────────────────────────────────────────
def get_narr_yx_slices(ds, bbox):
    """Trova il window NARR che copre il bbox (lat/lon 2D)."""
    lat = ds['lat'] if 'lat' in ds.coords else ds['latitude']
    lon = ds['lon'] if 'lon' in ds.coords else ds['longitude']
    lon180 = ((lon + 180) % 360) - 180
    mask = ((lat >= bbox['lat_min']) & (lat <= bbox['lat_max']) &
            (lon180 >= bbox['lon_min']) & (lon180 <= bbox['lon_max']))
    idx = np.argwhere(mask.values)
    if idx.size == 0:
        raise ValueError('Nessuna cella NARR nel bbox')
    y0, x0 = idx.min(axis=0)
    y1, x1 = idx.max(axis=0)
    halo = 3
    y0 = max(int(y0) - halo, 0); x0 = max(int(x0) - halo, 0)
    y1 = min(int(y1) + halo, mask.shape[0]-1); x1 = min(int(x1) + halo, mask.shape[1]-1)
    ydim, xdim = mask.dims
    return {ydim: slice(y0, y1+1), xdim: slice(x0, x1+1)}


def download_narr_year(year, prefix, out_name, yx_slices, overwrite=False):
    """Scarica subset NARR via OPeNDAP e salva netCDF locale."""
    out_file = NARR_RAW_DIR / f'narr_ext_{out_name}_{year}.nc'
    if out_file.exists() and not overwrite:
        return out_file
    url = f'{NARR_BASE}/{prefix}.{year}.nc'
    print(f'  Downloading {prefix}.{year} ...', end=' ', flush=True)
    t0 = time.time()
    ds = xr.open_dataset(url, **NARR_OPEN_KW)
    # trova il nome della variabile dati
    dv = [v for v in ds.data_vars if v.lower() not in {'lat','lon','lambert_conformal'}][0]
    da = ds[dv].isel(**yx_slices)
    # includi lat/lon 2D
    for c in ['lat','lon']:
        if c in ds:
            da = da.assign_coords({c: ds[c].isel(**yx_slices)})
    if 'lon' in da.coords:
        da = da.assign_coords(lon=((da['lon']+180)%360)-180)
    da = da.rename(out_name)
    da.load().to_netcdf(out_file, encoding={out_name: {'zlib': True, 'complevel': 4}})
    ds.close()
    print(f'OK ({time.time()-t0:.0f}s)  → {out_file.name}')
    return out_file


# Calcola yx_slices una volta dal primo anno disponibile
def init_narr_slices():
    ref_year = YEARS_AVAILABLE[0]
    url = f'{NARR_BASE}/air.2m.{ref_year}.nc'
    print(f'Apro NARR metadati da {url} ...')
    ds0 = xr.open_dataset(url, **NARR_OPEN_KW)
    slices = get_narr_yx_slices(ds0, NARR_BBOX)
    # Salva lat/lon 2D del subset
    narr_lat2d = ds0['lat'].isel(**slices).values.astype(np.float32)
    narr_lon2d = ((ds0['lon'].isel(**slices).values + 180) % 360 - 180).astype(np.float32)
    ds0.close()
    print(f'NARR subset: {narr_lat2d.shape}  '
          f'lat [{narr_lat2d.min():.2f}, {narr_lat2d.max():.2f}]  '
          f'lon [{narr_lon2d.min():.2f}, {narr_lon2d.max():.2f}]')
    return slices, narr_lat2d, narr_lon2d

print('Inizializzo NARR slices...')
YX_SLICES, NARR_LAT2D, NARR_LON2D = init_narr_slices()

In [ ]:
# ── DOWNLOAD NARR (esegui; skip se file già presente) ─────────────────────────
# Scarica solo gli anni per cui abbiamo SPORTLIS + anno precedente (buffer TM)
NARR_YEARS_NEEDED = sorted(set([min(YEARS_AVAILABLE)-1] + YEARS_AVAILABLE))
# Limita ad anni realmente disponibili in NARR (>= 1979)
NARR_YEARS_NEEDED = [y for y in NARR_YEARS_NEEDED if y >= 1979]

print(f'Anni NARR da scaricare: {len(NARR_YEARS_NEEDED)}  ({NARR_YEARS_NEEDED[0]}–{NARR_YEARS_NEEDED[-1]})')

for year in NARR_YEARS_NEEDED:
    for prefix, out_name in NARR_VARS_NEEDED.items():
        download_narr_year(year, prefix, out_name, YX_SLICES, overwrite=False)

print('Download NARR completato.')

## Sezione 3 — Pesi di regridding NARR→SPORTLIS (una-tantum)

In [ ]:
# ── 4. REGRID WEIGHTS (Inverse Distance Weighting, k=4 nearest NARR cells) ───
# I pesi vengono calcolati una volta e riutilizzati per ogni timestep/anno.

def build_regrid_weights(narr_lat2d, narr_lon2d, target_lat2d, target_lon2d, k=4):
    """
    Costruisce i pesi IDW per interpolare dalla griglia NARR (32 km, 2D lat/lon)
    alla griglia target SPORTLIS (3 km, 2D lat/lon regolare).

    Returns:
        inds   : (H_t*W_t, k) indici flat nella griglia NARR
        weights: (H_t*W_t, k) pesi IDW normalizzati
    """
    src = np.column_stack([narr_lat2d.ravel(), narr_lon2d.ravel()])
    tgt = np.column_stack([target_lat2d.ravel(), target_lon2d.ravel()])
    tree = cKDTree(src)
    dists, inds = tree.query(tgt, k=k)
    # IDW: w = 1/d^2; punti esatti → peso 1
    eps = 1e-10
    w = 1.0 / np.maximum(dists, eps)**2
    w /= w.sum(axis=1, keepdims=True)
    return inds.astype(np.int32), w.astype(np.float32)


def apply_regrid(data_flat, inds, weights):
    """Applica i pesi IDW a un campo NARR appiattito → campo SPORTLIS appiattito."""
    return (data_flat[inds] * weights).sum(axis=1)


# Griglia SPORTLIS estesa (lat/lon regolare 1D → 2D)
# La ricostruiamo dai file nc4 (usare il primo file disponibile)
def load_sportlis_grid(nc4_path):
    sp = nc.Dataset(nc4_path)
    try:
        lat2d = np.asarray(sp.variables['latitude'][:],  dtype=np.float32)  # (484, 698)
        lon2d = np.asarray(sp.variables['longitude'][:], dtype=np.float32)
    finally:
        sp.close()
    # lat/lon 1D dal bordo della griglia (assumiamo griglia quasi-regolare)
    lat_1d = lat2d[:, 0].astype(np.float32)
    lon_1d = lon2d[0, :].astype(np.float32)
    return lat2d, lon2d, lat_1d, lon_1d


first_nc4 = nc4_files[0]
print(f'Carico griglia SPORTLIS da {Path(first_nc4).name} ...')
SPORTLIS_LAT2D, SPORTLIS_LON2D, lat_1d, lon_1d = load_sportlis_grid(first_nc4)
print(f'  SPORTLIS shape: {SPORTLIS_LAT2D.shape}')
print(f'  lat: [{lat_1d.min():.3f}, {lat_1d.max():.3f}]  '
      f'lon: [{lon_1d.min():.3f}, {lon_1d.max():.3f}]')

# LAT_DESCENDING: True se lat[0] > lat[-1]
_LAT_DESCENDING = bool(lat_1d[0] > lat_1d[-1])
print(f'  LAT_DESCENDING={_LAT_DESCENDING}')

print('Calcolo pesi regridding NARR→SPORTLIS ...')
t0 = time.time()
REGRID_INDS, REGRID_WEIGHTS = build_regrid_weights(
    NARR_LAT2D, NARR_LON2D, SPORTLIS_LAT2D, SPORTLIS_LON2D, k=4
)
print(f'  OK in {time.time()-t0:.1f}s  inds shape: {REGRID_INDS.shape}')

H_S, W_S = SPORTLIS_LAT2D.shape

def regrid_batch(data_3d, inds=REGRID_INDS, weights=REGRID_WEIGHTS, chunk=None):
    """Regrid (T, Hn, Wn) -> (T, H_S, W_S) in batch per limitare il picco RAM.
    Peak RAM ≈ chunk * H_S*W_S * k * 4 bytes.
    Con chunk=REGRID_CHUNK=60: ~325 MB su dominio 484×698."""
    if chunk is None:
        chunk = REGRID_CHUNK
    T = data_3d.shape[0]
    flat = data_3d.reshape(T, -1)          # (T, Hn*Wn)
    out  = np.empty((T, H_S, W_S), dtype=np.float32)
    for t0 in range(0, T, chunk):
        t1 = min(t0 + chunk, T)
        # (chunk, H_S*W_S, k) → sum → (chunk, H_S*W_S)
        blk = (flat[t0:t1, inds] * weights).sum(axis=2)
        out[t0:t1] = blk.reshape(-1, H_S, W_S)
    return out

def orient(arr):
    return arr[::-1,:] if _LAT_DESCENDING else arr

## Sezione 4 — Static features (elevation, slope, aspect, TPI, canopy)

In [ ]:
# ── 5. STATIC FEATURES DAL FILE NC4 SPORTLIS ──────────────────────────────────

def load_sportlis_static_from_nc4(nc4_path):
    """Estrae elevation, slope, aspect dal gruppo 'static' della nc4."""
    sp = nc.Dataset(nc4_path)
    try:
        g = sp.groups['static']
        elev   = np.asarray(g.variables['elevation'][:], dtype=np.float32)
        slope  = np.asarray(g.variables['slope'][:],     dtype=np.float32)
        aspect = np.asarray(g.variables['aspect'][:],    dtype=np.float32)
    finally:
        sp.close()
    fill = -9999.0
    domain = (elev > fill) & (slope > fill) & (aspect > fill)
    elev   = np.where(domain, elev,   np.nan).astype(np.float32)
    slope  = np.where(domain, slope,  np.nan).astype(np.float32)
    aspect_rad = np.where(domain, aspect, np.nan).astype(np.float32)
    asp_sin = np.where(domain, np.sin(aspect_rad), np.nan).astype(np.float32)
    asp_cos = np.where(domain, np.cos(aspect_rad), np.nan).astype(np.float32)
    print(f'Static SPORTLIS: shape={elev.shape}  '
          f'dominio valido={domain.sum()}/{domain.size} ({100*domain.sum()/domain.size:.1f}%)')
    return dict(elevation=elev, slope=slope, aspect_sin=asp_sin,
                aspect_cos=asp_cos, domain=domain.astype(np.uint8))


if STATIC_FILE.exists():
    print(f'Static file già presente: {STATIC_FILE}')
    ds_static = xr.open_dataset(STATIC_FILE)
    static_dict = {v: ds_static[v].values.astype(np.float32)
                   for v in ['elevation','slope','aspect_sin','aspect_cos','domain']}
    ds_static.close()
else:
    static_dict = load_sportlis_static_from_nc4(first_nc4)
    # salva netCDF per uso futuro
    STATIC_FILE.parent.mkdir(parents=True, exist_ok=True)
    ds_st = xr.Dataset(
        {k: (('lat','lon'), v) for k, v in static_dict.items()},
        coords={'lat': lat_1d, 'lon': lon_1d,
                'latitude':  (('lat','lon'), SPORTLIS_LAT2D),
                'longitude': (('lat','lon'), SPORTLIS_LON2D)}
    )
    ds_st.to_netcdf(STATIC_FILE)
    print(f'Static salvato: {STATIC_FILE}')

domain2d = static_dict['domain'].astype(bool)
elev = static_dict['elevation']
print(f'Elevazione: min={np.nanmin(elev):.0f} m  max={np.nanmax(elev):.0f} m')

# Quick plot elevation
fig, ax = plt.subplots(figsize=(10,5))
ax.imshow(orient(elev), origin='lower', cmap='terrain',
          extent=[lon_1d.min(),lon_1d.max(),lat_1d.min(),lat_1d.max()])
ax.set_title('Elevation — dominio SPORTLIS esteso')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
plt.tight_layout(); plt.show()

In [ ]:
# ── 6. TPI (Topographic Position Index) ───────────────────────────────────────
elev_filled = np.where(np.isfinite(elev), elev, np.nanmedian(elev)).astype(np.float32)

tpi_layers = {}
for name, r in TPI_RADII.items():
    size = 2*r+1
    loc_mean = uniform_filter(elev_filled, size=size, mode='nearest').astype(np.float32)
    tpi_layers[name] = (elev_filled - loc_mean).astype(np.float32)
    print(f'  {name}: finestra {size}×{size}  range [{tpi_layers[name].min():.0f}, {tpi_layers[name].max():.0f}] m')

fig, axes = plt.subplots(1,2,figsize=(12,4))
ext = [lon_1d.min(),lon_1d.max(),lat_1d.min(),lat_1d.max()]
for ax,(name,arr) in zip(axes,tpi_layers.items()):
    vmax = np.percentile(np.abs(arr),99)
    ax.imshow(orient(arr), origin='lower', extent=ext, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# ── 7. CANOPY (Hansen GFC) ─────────────────────────────────────────────────────
# Dominio esteso copre più tile Hansen. Scarica i tile necessari:
# Il dominio (35-50 N, 125-104 W) è coperto principalmente da:
#   Hansen_GFC-2023-v1.11_treecover2000_50N_120W.tif  (copre 40-50N)
#   Hansen_GFC-2023-v1.11_treecover2000_40N_120W.tif  (copre 30-40N)
# e parti di tile _110W per la zona est.

HANSEN_TILES = [
    'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/Hansen_GFC-2023-v1.11_treecover2000_50N_120W.tif',
    'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/Hansen_GFC-2023-v1.11_treecover2000_40N_120W.tif',
    'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/Hansen_GFC-2023-v1.11_treecover2000_50N_110W.tif',
    'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/Hansen_GFC-2023-v1.11_treecover2000_40N_110W.tif',
]

def aggregate_hansen_tile(url, lat_1d, lon_1d, chunk_rows=512):
    """Legge un tile Hansen COG (windowed) e aggrega su griglia SPORTLIS."""
    try:
        import rasterio
        from rasterio.windows import from_bounds as window_from_bounds
    except ImportError:
        raise RuntimeError('pip install rasterio')

    H, W = len(lat_1d), len(lon_1d)
    lat_min, lat_max = float(lat_1d.min()), float(lat_1d.max())
    lon_min, lon_max = float(lon_1d.min()), float(lon_1d.max())
    pad = 0.05

    with rasterio.open(url) as src:
        win = window_from_bounds(lon_min-pad, lat_min-pad, lon_max+pad, lat_max+pad, src.transform)
        # controlla se il tile si sovrappone al dominio
        if win.width <= 0 or win.height <= 0:
            return None
        # clip al tile
        from rasterio.windows import intersection, Window
        full_win = Window(0, 0, src.width, src.height)
        try:
            win = intersection(win, full_win)
        except Exception:
            return None
        src_data = src.read(1, window=win)
        src_transform = src.window_transform(win)

    nrows_src, ncols_src = src_data.shape
    if nrows_src == 0 or ncols_src == 0:
        return None

    a = float(src_transform.a); e = float(src_transform.e)
    c0 = float(src_transform.c); f0 = float(src_transform.f)
    src_lon = c0 + a*(np.arange(ncols_src)+0.5)
    src_lat = f0 + e*(np.arange(nrows_src)+0.5)

    dlat = float(lat_1d[1]-lat_1d[0]) if _LAT_DESCENDING==False else float(lat_1d[0]-lat_1d[1])
    dlat = abs(dlat)
    dlon = abs(float(lon_1d[1]-lon_1d[0]))
    lat0 = float(lat_1d.min()) - dlat/2.0
    lon0 = float(lon_1d.min()) - dlon/2.0

    i_idx = np.floor((src_lat - lat0) / dlat).astype(np.int64)
    j_idx = np.floor((src_lon - lon0) / dlon).astype(np.int64)
    valid_i = (i_idx >= 0) & (i_idx < H)
    valid_j = (j_idx >= 0) & (j_idx < W)
    j_valid_cols = np.where(valid_j)[0]
    j_valid_idx  = j_idx[j_valid_cols]
    if len(j_valid_cols) == 0:
        return None

    canopy_sum = np.zeros((H, W), dtype=np.float64)
    canopy_cnt = np.zeros((H, W), dtype=np.int64)
    for r0 in range(0, nrows_src, chunk_rows):
        r1 = min(r0+chunk_rows, nrows_src)
        i_chunk = i_idx[r0:r1]
        good = np.where(valid_i[r0:r1])[0]
        if len(good) == 0: continue
        blk = src_data[r0:r1,:][good][:,j_valid_cols].astype(np.float32)
        i_blk = i_chunk[good]
        row_sum = np.zeros((H, len(j_valid_cols)), dtype=np.float64)
        row_cnt = np.zeros((H, len(j_valid_cols)), dtype=np.int64)
        np.add.at(row_sum, i_blk, blk)
        np.add.at(row_cnt, i_blk, 1)
        np.add.at(canopy_sum.T, j_valid_idx, row_sum.T)
        np.add.at(canopy_cnt.T, j_valid_idx, row_cnt.T)

    canopy_frac = (canopy_sum / np.maximum(canopy_cnt,1) / 100.0).astype(np.float32)
    canopy_frac = np.clip(canopy_frac, 0, 1)
    # dove cnt==0 non c'era sovrapposizione con questo tile
    canopy_frac[canopy_cnt == 0] = np.nan
    return canopy_frac


if CANOPY_FILE.exists():
    print(f'Canopy già presente: {CANOPY_FILE}')
    ds_cn = xr.open_dataset(CANOPY_FILE)
    canopy_fraction = ds_cn['canopy_fraction'].values.astype(np.float32)
    ds_cn.close()
else:
    print('Aggregazione canopy Hansen GFC (multi-tile)...')
    tiles = []
    for url in HANSEN_TILES:
        print(f'  tile: {Path(url).name}')
        t = aggregate_hansen_tile(url, lat_1d, lon_1d)
        if t is not None:
            tiles.append(t)
    # merge: media tra tile dove si sovrappongono
    stack = np.stack(tiles, axis=0)  # (N_tiles, H, W)
    canopy_fraction = np.nanmean(stack, axis=0).astype(np.float32)
    canopy_fraction = np.nan_to_num(canopy_fraction, nan=0.0)
    # salva
    xr.Dataset({'canopy_fraction': (('lat','lon'), canopy_fraction)},
               coords={'lat': lat_1d, 'lon': lon_1d},
               attrs={'source': 'Hansen GFC 2023 v1.11', 'units': 'fraction 0-1'}
               ).to_netcdf(CANOPY_FILE)
    print(f'  Canopy salvato: {CANOPY_FILE}')

print(f'canopy_fraction: shape={canopy_fraction.shape}  '
      f'mean={canopy_fraction.mean():.3f}  max={canopy_fraction.max():.3f}')

In [ ]:
# ── 8. ASSEMBLA topo_tensor (7 canali) ────────────────────────────────────────
topo_layers_raw = {}
for v in ['elevation','slope','aspect_sin','aspect_cos']:
    arr = static_dict[v].astype(np.float32)
    if v in ('aspect_sin','aspect_cos'):
        arr = np.where(np.isfinite(arr), arr, 0.0)
    else:
        arr = np.where(np.isnan(arr), np.nanmedian(arr), arr)
    topo_layers_raw[v] = arr
for name, arr in tpi_layers.items():
    topo_layers_raw[name] = arr
topo_layers_raw['canopy_fraction'] = canopy_fraction

assert list(topo_layers_raw.keys()) == TOPO_VARS, f'Mismatch: {list(topo_layers_raw.keys())}'

topo_stats = {}
topo_normalized = []
for name in TOPO_VARS:
    arr = topo_layers_raw[name]
    if name in ('aspect_sin','aspect_cos'):
        topo_stats[name] = (0.0, 1.0); topo_normalized.append(arr)
    elif name == 'canopy_fraction':
        m = float(np.nanmean(arr)); topo_stats[name] = (m, 1.0)
        topo_normalized.append((arr-m).astype(np.float32))
    else:
        m = float(np.nanmean(arr)); s = float(np.nanstd(arr)) or 1.0
        topo_stats[name] = (m, s)
        topo_normalized.append(((arr-m)/s).astype(np.float32))

topo_tensor = np.stack(topo_normalized, axis=0).astype(np.float32)
N_IN_CHANNELS = len(INPUT_VARS) + 2 + len(TOPO_VARS) + (2 if ADD_DOY else 0)
print(f'topo_tensor: {topo_tensor.shape}  N_IN_CHANNELS: {N_IN_CHANNELS}')

## Sezione 5 — Costruzione zarr per anno

In [ ]:
# ── 9. BUILD ZARR PER ANNO ─────────────────────────────────────────────────────
# Ogni zarr conterrà (time, lat, lon) con: precip, tair, swe_target_filled,
# swe_mask, is_snowy_time, latitude, longitude
# I timestep sono SOLO le ore NARR (0,3,6,9,12,15,18,21) presenti in entrambi.

SNOWY_THRESHOLD_MM = 5.0
SNOWY_MIN_PIXELS   = 200    # più alto per il dominio esteso
TIME_CHUNK_ZARR    = 480

def zarr_path(year):
    return ZARR_DIR / f'sportlis_narr_pp_{year}.zarr'


def build_year_zarr(year, overwrite=False):
    out_path = zarr_path(year)
    if out_path.exists() and not overwrite:
        print(f'  {year}: zarr già esistente, skip.')
        return
    if out_path.exists():
        shutil.rmtree(str(out_path))

    nc4_path = file_year_map[year]
    narr_tair_file   = NARR_RAW_DIR / f'narr_ext_tair_{year}.nc'
    narr_precip_file = NARR_RAW_DIR / f'narr_ext_precip_{year}.nc'

    for f in [narr_tair_file, narr_precip_file]:
        if not f.exists():
            raise FileNotFoundError(f'{f} non trovato — esegui prima il download NARR')

    print(f'  Costruisco zarr anno {year} ...')

    # 1) Leggi NARR
    ds_tair   = xr.open_dataset(narr_tair_file,   decode_times=True)
    ds_precip = xr.open_dataset(narr_precip_file, decode_times=True)
    narr_times = pd.to_datetime(ds_tair['time'].values)
    tair_arr   = ds_tair['tair'].values.astype(np.float32)    # (T_narr, y_narr, x_narr)
    precip_arr = ds_precip['precip'].values.astype(np.float32)
    ds_tair.close(); ds_precip.close()
    print(f'    NARR: {len(narr_times)} timestep  shape={tair_arr.shape}')

    # 2) Leggi SPORTLIS per l'anno
    sp = nc.Dataset(nc4_path)
    try:
        sp_ts = parse_sportlis_timestamps(sp)
        swe_raw_all = sp.variables['data']  # (T_sp, H, W, 1)
        mask_year = (sp_ts.year == year)
        sp_idx_year = np.where(mask_year)[0]
        sp_ts_year  = sp_ts[mask_year]
        print(f'    SPORTLIS anno {year}: {len(sp_ts_year)} ore')

        # 3) Intersezione temporale NARR ∩ SPORTLIS
        common, narr_pos, sp_pos = np.intersect1d(
            narr_times.values.astype('datetime64[ns]'),
            sp_ts_year.values.astype('datetime64[ns]'),
            return_indices=True
        )
        if len(common) == 0:
            print(f'    WARN: nessun timestep comune per {year}, skip.')
            return
        # ordina per sicurezza
        order = np.argsort(common)
        common = common[order]
        narr_pos_sorted = narr_pos[order]
        sp_pos_sorted   = sp_idx_year[sp_pos[order]]
        n_t = len(common)
        print(f'    Timestep comuni: {n_t}')

        # 4) Inizializza zarr
        ZARR_DIR.mkdir(parents=True, exist_ok=True)
        t_coord = pd.DatetimeIndex(common)
        enc = {'chunks': (TIME_CHUNK_ZARR, H_S, W_S)}
        enc1d = {'chunks': (TIME_CHUNK_ZARR,)}

        ds_out = xr.Dataset(
            {
                'precip':            (('time','lat','lon'), np.zeros((n_t,H_S,W_S), dtype=np.float32)),
                'tair':              (('time','lat','lon'), np.zeros((n_t,H_S,W_S), dtype=np.float32)),
                'swe_target_filled': (('time','lat','lon'), np.zeros((n_t,H_S,W_S), dtype=np.float32)),
                'swe_mask':          (('time','lat','lon'), np.zeros((n_t,H_S,W_S), dtype=np.uint8)),
                'is_snowy_time':     (('time',),            np.zeros((n_t,), dtype=np.uint8)),
                'latitude':          (('lat','lon'),         SPORTLIS_LAT2D),
                'longitude':         (('lat','lon'),         SPORTLIS_LON2D),
            },
            coords={'time': t_coord}
        )
        zarr_enc = {
            v: enc for v in ['precip','tair','swe_target_filled','swe_mask']
        }
        zarr_enc['is_snowy_time'] = enc1d
        ds_out.to_zarr(str(out_path), mode='w', encoding=zarr_enc)
        del ds_out

        # 5) Riempi a batch
        store = zarr.open(str(out_path), mode='r+')
        n_flat_narr = tair_arr.shape[1] * tair_arr.shape[2]

        BATCH = 240  # timestep per batch
        for a in range(0, n_t, BATCH):
            b = min(a+BATCH, n_t)
            ni = narr_pos_sorted[a:b]
            si = sp_pos_sorted[a:b]
            n = b - a

            # regrid NARR → SPORTLIS (vectorized batch, peak RAM ~325 MB)
            t_blk  = regrid_batch(tair_arr[ni])    # (n, H_S, W_S)
            pr_blk = regrid_batch(precip_arr[ni])

            # carica SWE
            swe_b = swe_raw_all[si, :, :, 0].astype(np.float32)  # (n, H, W)
            swe_valid   = (swe_b > -0.5) & np.isfinite(swe_b)
            swe_filled  = np.where(swe_valid, swe_b, 0.0).astype(np.float32)
            valid_mask  = (swe_valid & domain2d[None,:,:]).astype(np.uint8)

            snowy_px    = (swe_filled > SNOWY_THRESHOLD_MM) & (valid_mask > 0)
            snowy_cnt   = snowy_px.sum(axis=(1,2))
            is_snowy    = (snowy_cnt >= SNOWY_MIN_PIXELS).astype(np.uint8)

            store['precip'][a:b]            = pr_blk
            store['tair'][a:b]              = t_blk
            store['swe_target_filled'][a:b] = swe_filled
            store['swe_mask'][a:b]          = valid_mask
            store['is_snowy_time'][a:b]     = is_snowy

            if a % 2400 == 0:
                print(f'    batch t={a}..{b}/{n_t}  snowy={is_snowy.sum()}/{n}')

        zarr.consolidate_metadata(str(out_path))
        print(f'  Anno {year} zarr OK: {n_t} timestep  '
              f'snowy={int(store["is_snowy_time"][:].sum())}')
    finally:
        sp.close()


# Esegui per tutti gli anni
for year in YEARS_AVAILABLE:
    build_year_zarr(year, overwrite=False)

## Sezione 6 — Feature temporali estese (rolling windows)

In [ ]:
# ── 10. CAUSAL ROLLING + EXTENDED TM (identico al baseline) ───────────────────
def open_zarr_robust(path):
    p = str(path)
    for kw in [dict(consolidated=True), dict(consolidated=False),
                dict(consolidated=True, zarr_format=3), dict(consolidated=False, zarr_format=3)]:
        try:
            return xr.open_zarr(p, chunks={}, **kw)
        except TypeError: continue
        except Exception: continue
    return xr.open_zarr(p, chunks={})


def causal_rolling_sum(arr, window):
    arr = np.asarray(arr); cs = np.cumsum(arr, axis=0, dtype=arr.dtype)
    out = cs.copy(); out[window:] = cs[window:] - cs[:-window]
    return out

def causal_rolling_mean(arr, window):
    s = causal_rolling_sum(arr, window).astype(np.float64)
    T = arr.shape[0]
    d = np.minimum(np.arange(1,T+1), window).astype(np.float64)
    while d.ndim < arr.ndim: d = d[...,None]
    return (s/d).astype(arr.dtype)

def causal_rolling_max(arr, window):
    return maximum_filter1d(arr, size=window, axis=0,
                            origin=-((window-1)//2),
                            mode='constant', cval=np.finfo(arr.dtype).min
                           ).astype(arr.dtype)


def _get_prev_tail(year, var_name, n_steps):
    """Carica solo gli ultimi BUFFER_DAYS_CUTOFF giorni del prev-year
    (buffer trim — lezione SNODAS: non caricare tutto l'anno precedente)."""
    p = zarr_path(year-1)
    if not p.exists(): return None
    ds = open_zarr_robust(p)
    ts = pd.to_datetime(ds['time'].values)
    cutoff = pd.Timestamp(year=year-1, month=10, day=1) - pd.Timedelta(days=BUFFER_DAYS_CUTOFF)
    mask = ts >= cutoff
    if not mask.any():
        # fallback: ultimi n_steps
        start = max(0, ds.sizes['time'] - n_steps)
        buf = ds[var_name].isel(time=slice(start, None)).values.astype(np.float32)
    else:
        start = int(np.where(mask)[0][0])
        buf = ds[var_name].isel(time=slice(start, None)).values.astype(np.float32)
    ds.close()
    # tronca a n_steps se più lungo
    return buf[-n_steps:] if len(buf) > n_steps else buf

def _get_prev_oct_dec(year, var_name='precip'):
    p = zarr_path(year-1)
    if not p.exists(): return None
    ds = open_zarr_robust(p)
    ts = pd.to_datetime(ds['time'].values)
    mask = ts >= pd.Timestamp(year=year-1, month=10, day=1)
    if not mask.any(): ds.close(); return None
    arr = ds[var_name].isel(time=slice(int(np.where(mask)[0][0]), ds.sizes['time'])
                            ).values.astype(np.float32)
    ds.close(); return arr


def add_extended_temporal_features(ds_y, year):
    precip_y = ds_y['precip'].values.astype(np.float32)
    tair_y   = ds_y['tair'].values.astype(np.float32)

    pp = _get_prev_tail(year, 'precip', TEMPORAL_BUFFER)
    tp = _get_prev_tail(year, 'tair',   TEMPORAL_BUFFER)
    if pp is not None and tp is not None:
        pf = np.concatenate([pp, precip_y], axis=0)
        tf = np.concatenate([tp, tair_y],   axis=0)
        off = pp.shape[0]; del pp, tp
    else:
        pf = precip_y; tf = tair_y; off = 0

    p7   = causal_rolling_sum(pf,  WINDOW_7D )[off:].astype(np.float32)
    p30  = causal_rolling_sum(pf,  WINDOW_30D)[off:].astype(np.float32)
    p60  = causal_rolling_sum(pf,  WINDOW_60D)[off:].astype(np.float32)
    t30m = causal_rolling_mean(tf, WINDOW_30D)[off:].astype(np.float32)
    t30x = causal_rolling_max(tf,  WINDOW_30D)[off:].astype(np.float32)
    pdh  = np.maximum(tf - 273.15, 0.0).astype(np.float32)
    dd30 = causal_rolling_sum(pdh, WINDOW_30D)[off:].astype(np.float32)
    del pf, tf, pdh; gc.collect()

    # precip water-year-to-date
    times_y = pd.to_datetime(ds_y['time'].values)
    seed_arr = _get_prev_oct_dec(year)
    seed = seed_arr.sum(axis=0).astype(np.float32) if seed_arr is not None else np.zeros(precip_y.shape[1:], np.float32)
    oct1 = pd.Timestamp(year=year, month=10, day=1)
    mask_oct = times_y >= oct1
    oct1_idx = int(np.where(mask_oct)[0][0]) if mask_oct.any() else precip_y.shape[0]
    pwytd = np.empty_like(precip_y)
    if oct1_idx > 0:
        pwytd[:oct1_idx] = seed[None] + np.cumsum(precip_y[:oct1_idx], axis=0)
    if oct1_idx < precip_y.shape[0]:
        pwytd[oct1_idx:] = np.cumsum(precip_y[oct1_idx:], axis=0)

    ds_y['precip_7d']      = (('time','lat','lon'), p7)
    ds_y['precip_30d']     = (('time','lat','lon'), p30)
    ds_y['precip_60d']     = (('time','lat','lon'), p60)
    ds_y['precip_wytd']    = (('time','lat','lon'), pwytd.astype(np.float32))
    ds_y['tair_30d_mean']  = (('time','lat','lon'), t30m)
    ds_y['tair_30d_max']   = (('time','lat','lon'), t30x)
    ds_y['degree_day_30d'] = (('time','lat','lon'), dd30)
    return ds_y


def snowy_indices(ds):
    if 'is_snowy_time' not in ds: return np.arange(ds.sizes['time'])
    return np.where(np.asarray(ds['is_snowy_time'].values) > 0)[0]


def load_year(year, snowy_only=True, time_stride=1):
    ds = open_zarr_robust(zarr_path(year))
    ds = add_extended_temporal_features(ds, year)
    if snowy_only:
        idx = snowy_indices(ds)
        if time_stride > 1: idx = idx[::time_stride]
        ds = ds.isel(time=idx)
        print(f'  {year}: {len(idx)} snowy ts (stride={time_stride})')
    elif time_stride > 1:
        ds = ds.isel(time=slice(None, None, time_stride))
    return ds

In [ ]:
# ── PRE-BAKE MEMMAP (lezione SNODAS: 50-100x più veloce di zarr-per-getitem) ──
# Costruisce un file .npy per anno con shape (T_snowy, N_FEAT+1, H_S, W_S)
# dove i canali sono [precip_7d, precip_30d, precip_60d, precip_wytd,
#                      tair_30d_mean, tair_30d_max, degree_day_30d, swe_target_filled]
# Successivamente il Dataset legge con mmap_mode='r' solo il patch richiesto.

import numpy.lib.format as np_format
MEMMAP_DIR.mkdir(parents=True, exist_ok=True)
N_FEAT = len(INPUT_VARS)  # 7

def build_memmap_year(year, overwrite=False):
    out_mm   = MEMMAP_DIR / f'y{year}_feat.npy'
    out_mask = MEMMAP_DIR / f'y{year}_mask.npy'
    out_info = MEMMAP_DIR / f'y{year}_info.json'
    if out_mm.exists() and not overwrite:
        print(f'  {year}: memmap già presente ({out_mm.name}), skip.')
        return out_mm, out_mask

    print(f'  {year}: carico zarr + aggiungo TM features...')
    ds = open_zarr_robust(zarr_path(year))
    ds = add_extended_temporal_features(ds, year)
    if USE_SNOWY_ONLY:
        idx = snowy_indices(ds)
        if TIME_STRIDE_TRAIN > 1:
            idx = idx[::TIME_STRIDE_TRAIN]
        ds = ds.isel(time=idx)
    T_v = ds.sizes['time']
    if T_v == 0:
        print(f'  {year}: 0 snowy timestep, skip.')
        return None, None

    # Feature array: (T_v, N_FEAT+1, H_S, W_S)
    arr = np_format.open_memmap(str(out_mm), mode='w+', dtype=np.float32,
                                shape=(T_v, N_FEAT + 1, H_S, W_S))
    mask_arr = np_format.open_memmap(str(out_mask), mode='w+', dtype=np.uint8,
                                     shape=(T_v, H_S, W_S))

    print(f'  {year}: scrittura memmap ({T_v} ts, shape {arr.shape})...')
    all_vars = INPUT_VARS + [TARGET_VAR]
    vars_need = all_vars + [MASK_VAR]
    ds_loaded = ds[vars_need].load()

    for ci, v in enumerate(all_vars):
        print(f'    {v}...', end=' ', flush=True)
        arr[:, ci] = ds_loaded[v].values.astype(np.float32)
        print('OK')
    mask_arr[:] = ds_loaded[MASK_VAR].values.astype(np.uint8)
    arr.flush(); mask_arr.flush()

    times = pd.to_datetime(ds_loaded['time'].values)
    info = {'year': year, 'T': T_v, 'H': H_S, 'W': W_S, 'N_FEAT': N_FEAT,
            'vars': all_vars, 'time_start': str(times[0]), 'time_end': str(times[-1])}
    with open(out_info, 'w') as fj: json.dump(info, fj)

    size_gb = out_mm.stat().st_size / 1024**3
    print(f'  {year}: OK  {T_v} ts  {size_gb:.2f} GB → {out_mm.name}')
    del ds_loaded, ds; gc.collect()
    return out_mm, out_mask


if USE_MEMMAP:
    print('Pre-bake memmap per tutti gli anni...')
    for year in YEARS_AVAILABLE:
        build_memmap_year(year, overwrite=False)
    print('Pre-bake completato.')
else:
    print('USE_MEMMAP=False: salto pre-bake, userò zarr-per-getitem (più lento).')

## Sezione 7 — Modello U-Net + Loss + Dataset

In [ ]:
# ── 11. SWEZarrDataset (inline, identico a swe_dataset.py) ───────────────────
class SWEZarrDataset(Dataset):
    def __init__(self, ds, input_vars, target_var, mask_var, mean_ds, std_ds,
                 topo_tensor, augment=False, add_doy=False, patch_size=None,
                 lat_1d=None, lon_1d=None):
        self.ds = ds; self.input_vars = input_vars
        self.target_var = target_var; self.mask_var = mask_var
        self.mean_ds = mean_ds; self.std_ds = std_ds
        self.n = ds.sizes['time']; self.augment = augment
        self.add_doy = add_doy; self.patch_size = patch_size
        self.topo_tensor = topo_tensor.astype(np.float32)
        l1 = np.asarray(lat_1d if lat_1d is not None else ds['lat'].values, np.float32)
        o1 = np.asarray(lon_1d if lon_1d is not None else ds['lon'].values, np.float32)
        lat2d, lon2d = np.meshgrid(l1, o1, indexing='ij')
        self.lat2d = lat2d.astype(np.float32); self.lon2d = lon2d.astype(np.float32)
        self.lat_mean = float(self.lat2d.mean()); self.lat_std = float(self.lat2d.std()) or 1.0
        self.lon_mean = float(self.lon2d.mean()); self.lon_std = float(self.lon2d.std()) or 1.0
        if self.add_doy:
            times = pd.to_datetime(ds['time'].values)
            doy = times.dayofyear.values.astype(np.float32)
            angle = 2.0*np.pi*doy/366.0
            self.doy_sin = np.sin(angle).astype(np.float32)
            self.doy_cos = np.cos(angle).astype(np.float32)

    def __len__(self): return self.n

    def _random_patch(self, x, y, m):
        H, W = x.shape[-2], x.shape[-1]
        ph = pw = self.patch_size
        if H < ph or W < pw: return x, y, m
        for _ in range(8):
            i = np.random.randint(0, H-ph+1); j = np.random.randint(0, W-pw+1)
            if m[..., i:i+ph, j:j+pw].sum() > 0.10*ph*pw: break
        return x[...,i:i+ph,j:j+pw], y[...,i:i+ph,j:j+pw], m[...,i:i+ph,j:j+pw]

    def __getitem__(self, idx):
        s = self.ds.isel(time=idx).load()
        x_list = []
        for v in self.input_vars:
            arr = s[v].values.astype(np.float32)
            arr = (arr - self.mean_ds[v].values.astype(np.float32)) / self.std_ds[v].values.astype(np.float32)
            x_list.append(arr)
        x_list.append((self.lat2d - self.lat_mean) / self.lat_std)
        x_list.append((self.lon2d - self.lon_mean) / self.lon_std)
        x = np.concatenate([np.stack(x_list, axis=0), self.topo_tensor], axis=0)
        if self.add_doy:
            H, W = self.lat2d.shape
            x = np.concatenate([x,
                                 np.full((1,H,W), self.doy_sin[idx], np.float32),
                                 np.full((1,H,W), self.doy_cos[idx], np.float32)], axis=0)
        y = np.log1p(np.maximum(s[self.target_var].values.astype(np.float32), 0.0))
        m = s[self.mask_var].values.astype(np.float32)
        y = y[None]; m = m[None]
        if self.patch_size is not None: x, y, m = self._random_patch(x, y, m)
        if self.augment:
            n_f = len(self.input_vars)
            x[:n_f] += np.random.randn(n_f, *x.shape[1:]).astype(np.float32)*0.02
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(m)

# ── SWEMemmapDataset (lezione SNODAS: usa quando USE_MEMMAP=True) ─────────────
class SWEMemmapDataset(Dataset):
    """Legge features da file .npy con mmap_mode='r'.
    Compatibile con num_workers>0. Gestisce n_patches_per_epoch per
    controllare il numero di campioni per epoch senza iterare tutto il dataset."""
    def __init__(self, years, mean_ds, std_ds, topo_tensor,
                 patch_size=PATCH_SIZE, n_patches=N_PATCHES_PER_EPOCH,
                 augment=False, lat_1d=None, lon_1d=None):
        self.patch = patch_size
        self.n     = n_patches
        self.augment = augment
        self.topo  = topo_tensor.astype(np.float32)   # (C_topo, H, W)

        # Normalizzazione
        self.mean = np.stack([mean_ds[v].values.astype(np.float32) for v in INPUT_VARS], 0)
        self.std  = np.stack([std_ds[v].values.astype(np.float32)  for v in INPUT_VARS], 0)

        # lat/lon normalizzati
        l1 = np.asarray(lat_1d, np.float32); o1 = np.asarray(lon_1d, np.float32)
        lat2d, lon2d = np.meshgrid(l1, o1, indexing='ij')
        self.lat_norm = ((lat2d - lat2d.mean()) / (lat2d.std() or 1.0)).astype(np.float32)
        self.lon_norm = ((lon2d - lon2d.mean()) / (lon2d.std() or 1.0)).astype(np.float32)

        # Carica i file memmap (mmap_mode='r' → accesso lazy al disco)
        self.mm_feat = {}; self.mm_mask = {}; self.mm_T = {}
        for y in years:
            fm = MEMMAP_DIR / f'y{y}_feat.npy'
            mk = MEMMAP_DIR / f'y{y}_mask.npy'
            if not fm.exists() or not mk.exists():
                print(f'  WARN: memmap mancante per {y}, skip')
                continue
            arr = np.load(str(fm), mmap_mode='r')
            self.mm_feat[y] = fm
        self.mm_mask = {y: MEMMAP_DIR/f'y{y}_mask.npy'
                        for y in years if (MEMMAP_DIR/f'y{y}_mask.npy').exists()}
        self.year_list = [y for y in years if y in self.mm_feat]
        self.T_cumsum  = []
        total = 0
        for y in self.year_list:
            a = np.load(str(self.mm_feat[y]), mmap_mode='r')
            total += a.shape[0]
            self.T_cumsum.append(total)
        self.T_total = total
        print(f'SWEMemmapDataset: {len(self.year_list)} anni  '              f'{self.T_total} ts  n_patches/epoch={n_patches}')

    def __len__(self): return self.n

    def _get_random_sample(self):
        """Campiona casualmente un (t, year) da tutti gli anni disponibili."""        t_global = np.random.randint(0, self.T_total)
        for yi, y in enumerate(self.year_list):
            if t_global < self.T_cumsum[yi]:
                t_local = t_global - (self.T_cumsum[yi-1] if yi>0 else 0)
                return y, t_local
        return self.year_list[-1], 0

    def __getitem__(self, _):
        y, ti = self._get_random_sample()
        feat = np.load(str(self.mm_feat[y]), mmap_mode='r')   # (T, N_FEAT+1, H, W)
        mask = np.load(str(self.mm_mask[y]), mmap_mode='r')   # (T, H, W)

        # Estrai snapshot e random patch
        H, W = feat.shape[2], feat.shape[3]
        ph = pw = self.patch
        for _ in range(8):
            i = np.random.randint(0, max(1, H-ph+1))
            j = np.random.randint(0, max(1, W-pw+1))
            if mask[ti, i:i+ph, j:j+pw].sum() > 0.10*ph*pw: break

        f_patch = feat[ti, :, i:i+ph, j:j+pw].copy()   # (N_FEAT+1, ph, pw)
        m_patch = mask[ti, i:i+ph, j:j+pw].copy()       # (ph, pw)

        # Normalizza feature
        x_feat = (f_patch[:N_FEAT] - self.mean.reshape(-1,1,1)) / self.std.reshape(-1,1,1)

        # lat/lon coord
        x_lat = self.lat_norm[i:i+ph, j:j+pw][None]
        x_lon = self.lon_norm[i:i+ph, j:j+pw][None]

        # topo
        x_topo = self.topo[:, i:i+ph, j:j+pw]

        x = np.concatenate([x_feat, x_lat, x_lon, x_topo], axis=0).astype(np.float32)

        y_val = np.log1p(np.maximum(f_patch[N_FEAT:N_FEAT+1], 0.0))  # (1, ph, pw)
        m_val = m_patch[None].astype(np.float32)                       # (1, ph, pw)

        if self.augment:
            x[:N_FEAT] += np.random.randn(N_FEAT, ph, pw).astype(np.float32) * 0.02

        return torch.from_numpy(x), torch.from_numpy(y_val), torch.from_numpy(m_val)

In [ ]:
# ── 12. U-Net (identica al baseline TPI+canopy) ───────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        g = min(8, out_ch)
        while out_ch % g != 0: g -= 1
        layers = [nn.Conv2d(in_ch,out_ch,3,padding=1), nn.GroupNorm(g,out_ch), nn.GELU(),
                  nn.Conv2d(out_ch,out_ch,3,padding=1), nn.GroupNorm(g,out_ch), nn.GELU()]
        if dropout > 0: layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def pad_to_mult(x, m=8):
    h,w = x.shape[-2],x.shape[-1]
    H = ((h+m-1)//m)*m; W = ((w+m-1)//m)*m
    ph = H-h; pw = W-w
    pl = pw//2; pr = pw-pl; pt = ph//2; pb = ph-pt
    return F.pad(x,(pl,pr,pt,pb),mode='reflect'),(pl,pr,pt,pb)

def unpad(x, p):
    pl,pr,pt,pb = p; H,W = x.shape[-2],x.shape[-1]
    return x[..., pt:H-pb if pb>0 else H, pl:W-pr if pr>0 else W]

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=1, base=32, dropout=0.1):
        super().__init__()
        self.e1=DoubleConv(in_channels,base); self.p1=nn.MaxPool2d(2)
        self.e2=DoubleConv(base,base*2);      self.p2=nn.MaxPool2d(2)
        self.e3=DoubleConv(base*2,base*4);    self.p3=nn.MaxPool2d(2)
        self.bn=DoubleConv(base*4,base*8,dropout=dropout)
        self.u1=nn.ConvTranspose2d(base*8,base*4,2,stride=2); self.d1=DoubleConv(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2); self.d2=DoubleConv(base*4,base*2)
        self.u3=nn.ConvTranspose2d(base*2,base,2,stride=2);   self.d3=DoubleConv(base*2,base)
        self.out=nn.Conv2d(base,out_channels,1)
    def forward(self,x):
        xp,pads = pad_to_mult(x)
        e1=self.e1(xp); e2=self.e2(self.p1(e1)); e3=self.e3(self.p2(e2))
        b=self.bn(self.p3(e3))
        d=self.d1(torch.cat([self.u1(b),e3],1))
        d=self.d2(torch.cat([self.u2(d),e2],1))
        d=self.d3(torch.cat([self.u3(d),e1],1))
        return F.softplus(unpad(self.out(d),pads))

In [ ]:
# ── 13. LOSS + HELPERS ────────────────────────────────────────────────────────
def masked_hybrid_loss(pred, target, mask, eps=1e-8):
    per_log = F.smooth_l1_loss(pred, target, reduction='none', beta=LOSS_LOG_BETA)
    pm = torch.expm1(torch.clamp(pred,   0, 10))
    tm = torch.expm1(torch.clamp(target, 0, 10))
    per_mm = F.smooth_l1_loss(pm, tm, reduction='none', beta=LOSS_MM_BETA) / 100.0
    per_h  = (1-LOSS_MM_WEIGHT)*per_log + LOSS_MM_WEIGHT*per_mm
    w  = 1.0 + LOSS_TARGET_ALPHA*torch.clamp(target/6.0, 0, 1.5)
    wm = w * mask
    data = (per_h*wm).sum() / (wm.sum()+eps)
    bias = torch.abs(((pm-tm)*mask).sum() / mask.sum().clamp(min=1)) / 100.0
    return data + LOSS_BIAS_WEIGHT*bias


def compute_norm_stats(train_years, spatial_stride=8):
    """Calcola mean/std dai memmap pre-baked con subsampling spaziale.
    Ogni anno: apre memmap (T_snowy, N_FEAT+1, H, W) e campiona
    ogni spatial_stride pixel -> peak RAM << 1 GB per anno.
    Fallback su zarr se memmap non disponibile.
    """
    n_feat = len(INPUT_VARS)
    n_tot = 0
    sum_v  = np.zeros(n_feat, dtype=np.float64)
    sum_v2 = np.zeros(n_feat, dtype=np.float64)

    for y in train_years:
        mm_path  = MEMMAP_DIR / f'y{y}_feat.npy'
        msk_path = MEMMAP_DIR / f'y{y}_mask.npy'
        if mm_path.exists():
            mm  = np.lib.format.open_memmap(str(mm_path),  mode='r')
            msk = np.lib.format.open_memmap(str(msk_path), mode='r')
            # mm shape: (T, N_FEAT+1, H, W) — canali 0..N_FEAT-1 = INPUT_VARS
            # subsample spaziale: prende ogni spatial_stride pixel
            feat = mm[:, :n_feat, ::spatial_stride, ::spatial_stride]  # (T, F, h, w)
            mask = (msk[:, 0, ::spatial_stride, ::spatial_stride] if msk.ndim == 4
                else msk[:, ::spatial_stride, ::spatial_stride])  # (T, h, w)
            T, F, h, w = feat.shape
            feat_flat = feat.reshape(T, F, -1).astype(np.float64)      # (T, F, h*w)
            mask_flat = mask.reshape(T, -1).astype(bool)               # (T, h*w)
            for f in range(F):
                vals = feat_flat[:, f, :][mask_flat]   # solo pixel validi
                sum_v[f]  += vals.sum()
                sum_v2[f] += (vals**2).sum()
                n_tot_f    = vals.size
                if f == 0: n_tot += n_tot_f
            del mm, msk, feat, mask, feat_flat, mask_flat
        else:
            # Fallback zarr: carica 1 variabile alla volta in chunk temporali
            print(f'  {y}: memmap non trovato, uso zarr (piu lento)')
            ds = load_year(y, snowy_only=USE_SNOWY_ONLY, time_stride=1)
            n_y = int(ds.sizes['time'])
            if n_y == 0: continue
            for fi, v in enumerate(INPUT_VARS):
                arr = ds[v].values.astype(np.float64)  # (T, H, W)
                sum_v[fi]  += arr.sum()
                sum_v2[fi] += (arr**2).sum()
                if fi == 0: n_tot += arr.size
                del arr

        print(f'  {y}: stats accumulate  (n_tot={n_tot:,})')

    mean_arr = sum_v  / n_tot
    std_arr  = np.sqrt(np.maximum(sum_v2/n_tot - mean_arr**2, 1e-12))

    # Ricostruisce xr.Dataset compatibile col resto del codice
    mean_ds = xr.Dataset({v: xr.DataArray(float(mean_arr[i])) for i,v in enumerate(INPUT_VARS)})
    std_ds  = xr.Dataset({v: xr.DataArray(float(max(std_arr[i], 1e-6))) for i,v in enumerate(INPUT_VARS)})
    return mean_ds, std_ds, n_tot


# ── DEVICE ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():   device = torch.device('mps')
elif torch.cuda.is_available():          device = torch.device('cuda')
else:                                    device = torch.device('cpu')
print(f'Device: {device}')


## Sezione 8 — LOYO Training (stride configurabile)

Con `LOYO_STEP=5` (default) vengono usati come test gli anni `{first_year+4, first_year+9, ...}`, 
ottenendo ~8 fold su 40 anni invece di 40 fold. Impostare `LOYO_STEP=1` per un LOYO completo.

In [ ]:
# ── 14. LOYO FOLDS ────────────────────────────────────────────────────────────
# Test years: ogni LOYO_STEP anni; val: anno successivo al test (ciclico)
test_years_all = YEARS_AVAILABLE[::LOYO_STEP]
LOYO_FOLDS = []
for ty in test_years_all:
    # val: anno immediatamente dopo il test (se disponibile)
    idx_t = YEARS_AVAILABLE.index(ty)
    vy = YEARS_AVAILABLE[(idx_t+1) % len(YEARS_AVAILABLE)]
    LOYO_FOLDS.append({'test': ty, 'val': vy})

print(f'LOYO_STEP={LOYO_STEP}  →  {len(LOYO_FOLDS)} fold:')
for f in LOYO_FOLDS:
    tr = [y for y in YEARS_AVAILABLE if y not in (f['test'], f['val'])]
    print(f"  test={f['test']}  val={f['val']}  train({len(tr)})={tr[0]}..{tr[-1]}")

In [ ]:
# ── 15. TRAINING + EVAL LOOP ──────────────────────────────────────────────────
# Resume logic: dopo ogni epoch salva un checkpoint "resume" con model +
# optimizer + scheduler + epoch + best_val + stale.
# Al riavvio rileva il resume checkpoint e continua dall'epoch interrotta.
# Il resume checkpoint viene rimosso a fine training (o early stop).

def resume_ckpt_path(ty):
    return OUTPUT_DIR / f'resume_unet_extended_LOYO_test{ty}.pt'


def eval_year(ds_test, ckpt_path, mean_ds, std_ds):
    ds_eager = ds_test[INPUT_VARS+[TARGET_VAR,MASK_VAR]].load()
    dset = SWEZarrDataset(ds_eager, INPUT_VARS, TARGET_VAR, MASK_VAR,
                          mean_ds, std_ds, topo_tensor, augment=False,
                          add_doy=ADD_DOY, patch_size=None, lat_1d=lat_1d, lon_1d=lon_1d)
    loader = DataLoader(dset, batch_size=2, shuffle=False, num_workers=0)
    m = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    m.eval()
    sa=sb=sq=sn=0.0; all_p=[]; all_t=[]
    with torch.no_grad():
        for Xb,yb,mb in loader:
            pred = m(Xb.to(device)).cpu().numpy()[:,0]
            true = yb.numpy()[:,0]; msk = mb.numpy()[:,0]>0
            pred = np.clip(pred,0,10)
            pm = np.expm1(pred); tm = np.expm1(true)
            for b in range(pm.shape[0]):
                mk = msk[b]
                if mk.sum()==0: continue
                er = pm[b][mk]-tm[b][mk]
                sa+=np.abs(er).sum(); sb+=er.sum(); sq+=(er**2).sum(); sn+=mk.sum()
                all_p.append(pm[b][mk]); all_t.append(tm[b][mk])
    p=np.concatenate(all_p); t=np.concatenate(all_t)
    mae=sa/sn; bias=sb/sn; rmse=np.sqrt(sq/sn)
    r2=float(np.corrcoef(p,t)[0,1]**2)
    nse=float(1-np.sum((p-t)**2)/np.sum((t-t.mean())**2))
    del ds_eager,dset,loader,m; gc.collect()
    return dict(mae=mae,bias=bias,rmse=rmse,r2=r2,nse=nse,n_pix=int(sn))


fold_results=[]

for fi,fold in enumerate(LOYO_FOLDS):
    ty=fold['test']; vy=fold['val']
    train_yrs=[y for y in YEARS_AVAILABLE if y not in (ty,vy)]
    ckpt     = OUTPUT_DIR / CHECKPOINT_TEMPLATE.format(test=ty)
    res_ckpt = resume_ckpt_path(ty)

    print(f'\n{"="*65}')
    print(f' FOLD {fi+1}/{len(LOYO_FOLDS)}  test={ty}  val={vy}  train({len(train_yrs)}): {train_yrs[0]}..{train_yrs[-1]}')

    # ── Decide se fare training ────────────────────────────────────────────────
    training_done = ckpt.exists() and not res_ckpt.exists()

    # ── Welford stats con cache su disco ─────────────────────────────────────
    stats_cache = OUTPUT_DIR / f'norm_stats_test{ty}.npz'
    if stats_cache.exists():
        sc = np.load(str(stats_cache), allow_pickle=True)
        mean_arr = sc['mean_arr']; std_arr = sc['std_arr']
        mean_ds = xr.Dataset({v: xr.DataArray(float(mean_arr[i])) for i,v in enumerate(INPUT_VARS)})
        std_ds  = xr.Dataset({v: xr.DataArray(float(std_arr[i]))  for i,v in enumerate(INPUT_VARS)})
        print(f' Welford stats caricate da cache ({stats_cache.name})')
    else:
        print(f' Welford stats su {len(train_yrs)} anni...')
        mean_ds, std_ds, n_tr = compute_norm_stats(train_yrs)
        np.savez(str(stats_cache),
                 mean_arr=np.array([float(mean_ds[v].values) for v in INPUT_VARS]),
                 std_arr =np.array([float(std_ds[v].values)  for v in INPUT_VARS]))
        print(f'  n_train_snowy={n_tr}  (cache salvata)')

    if training_done:
        print(f' Best checkpoint già presente e nessun resume pendente — solo eval.')
    else:

        # ── Dataset + DataLoader ──────────────────────────────────────────────
        if USE_MEMMAP:
            print(f' SWEMemmapDataset ({len(train_yrs)} anni, {N_PATCHES_PER_EPOCH} patch/epoch)...')
            tr_ds = SWEMemmapDataset(train_yrs, mean_ds, std_ds, topo_tensor,
                                     patch_size=PATCH_SIZE, n_patches=N_PATCHES_PER_EPOCH,
                                     augment=AUGMENT_NOISE, lat_1d=lat_1d, lon_1d=lon_1d)
            ds_val = load_year(vy, snowy_only=USE_SNOWY_ONLY, time_stride=1)
            ds_val = ds_val[INPUT_VARS+[TARGET_VAR,MASK_VAR]].load()
            vl_ds  = SWEZarrDataset(ds_val, INPUT_VARS, TARGET_VAR, MASK_VAR,
                                     mean_ds, std_ds, topo_tensor,
                                     augment=False, add_doy=ADD_DOY, patch_size=None,
                                     lat_1d=lat_1d, lon_1d=lon_1d)
            nw_train = min(4, max(NUM_WORKERS, 2))
        else:
            print(' Zarr path...')
            dsets_tr=[load_year(y,snowy_only=USE_SNOWY_ONLY,time_stride=TIME_STRIDE_TRAIN)
                      for y in train_yrs]
            ds_tr=xr.concat(dsets_tr,dim='time',data_vars='minimal',
                             coords='minimal',compat='override')
            ds_val=load_year(vy,snowy_only=USE_SNOWY_ONLY,time_stride=1)
            vars_need=INPUT_VARS+[TARGET_VAR,MASK_VAR]
            ds_tr=ds_tr[vars_need].load(); ds_val=ds_val[vars_need].load()
            tr_ds=SWEZarrDataset(ds_tr,INPUT_VARS,TARGET_VAR,MASK_VAR,mean_ds,std_ds,topo_tensor,
                                  augment=AUGMENT_NOISE,add_doy=ADD_DOY,patch_size=PATCH_SIZE,
                                  lat_1d=lat_1d,lon_1d=lon_1d)
            vl_ds=SWEZarrDataset(ds_val,INPUT_VARS,TARGET_VAR,MASK_VAR,mean_ds,std_ds,topo_tensor,
                                  augment=False,add_doy=ADD_DOY,patch_size=None,
                                  lat_1d=lat_1d,lon_1d=lon_1d)
            nw_train = NUM_WORKERS

        tr_ld=DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True,
                         num_workers=nw_train,pin_memory=(nw_train>0))
        vl_ld=DataLoader(vl_ds,batch_size=2,shuffle=False,num_workers=0)

        # ── Modello + ottimizzatore ───────────────────────────────────────────
        model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
        opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)

        # ── RESUME: ricarica stato se il checkpoint di resume esiste ──────────
        start_epoch = 0
        best_val    = float('inf')
        stale       = 0

        if res_ckpt.exists():
            print(f' *** RESUME da {res_ckpt.name} ***')
            state = torch.load(res_ckpt, map_location=device)
            model.load_state_dict(state['model'])
            opt.load_state_dict(state['optimizer'])
            sched.load_state_dict(state['scheduler'])
            start_epoch = state['epoch'] + 1   # riparte dall'epoch successiva
            best_val    = state['best_val']
            stale       = state['stale']
            print(f'   → riparte da epoch {start_epoch+1}/{EPOCHS}  ')
            print(f'     best_val={best_val:.4f}  stale={stale}/{PATIENCE}')
        elif ckpt.exists():
            # best checkpoint esiste ma c'è anche un resume: ricarica i pesi migliori
            print(f' Best ckpt trovato senza resume — carico pesi e continuo ')
            model.load_state_dict(torch.load(ckpt, map_location=device))

        # ── Training loop ─────────────────────────────────────────────────────
        for ep in range(start_epoch, EPOCHS):
            model.train(); rn=0.0; nb_b=0
            for ib,(Xb,yb,mb) in enumerate(tr_ld):
                Xb=Xb.to(device); yb=yb.to(device); mb=mb.to(device)
                pred=model(Xb)
                if torch.isnan(pred).any(): continue
                loss=masked_hybrid_loss(pred,yb,mb)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step(); rn+=loss.item(); nb_b+=1
            tr_loss=rn/max(nb_b,1)

            model.eval(); rv=0.0; nv=0
            with torch.no_grad():
                for Xb,yb,mb in vl_ld:
                    pred=model(Xb.to(device))
                    if torch.isnan(pred).any(): continue
                    rv+=masked_hybrid_loss(pred,yb.to(device),mb.to(device)).item(); nv+=1
            vl_loss=rv/max(nv,1)
            sched.step(vl_loss)

            improved = vl_loss < best_val
            if improved:
                best_val=vl_loss; stale=0
                torch.save(model.state_dict(), ckpt)
            else:
                stale+=1

            print(f'  E{ep+1:02d}/{EPOCHS} train={tr_loss:.4f} val={vl_loss:.4f} ')
            print(f'          lr={opt.param_groups[0]["lr"]:.2e}  ')
            print(f'          {"*best saved*" if improved else f"stale {stale}/{PATIENCE}"}')

            # ── Salva resume checkpoint dopo ogni epoch ───────────────────────
            torch.save({
                'epoch'    : ep,
                'model'    : model.state_dict(),
                'optimizer': opt.state_dict(),
                'scheduler': sched.state_dict(),
                'best_val' : best_val,
                'stale'    : stale,
                'train_yrs': train_yrs,
                'test_year': ty,
            }, res_ckpt)

            if stale >= PATIENCE:
                print(f'  Early stop (patience={PATIENCE}).')
                break

        # ── Fine training: rimuovi resume checkpoint ──────────────────────────
        if res_ckpt.exists():
            res_ckpt.unlink()
            print(f' Resume checkpoint rimosso ({res_ckpt.name}).')

        del tr_ld, vl_ld, model, opt, sched; gc.collect()
        if device.type=='mps' and hasattr(torch.mps,'empty_cache'): torch.mps.empty_cache()
        elif device.type=='cuda': torch.cuda.empty_cache()

    # ── EVAL ──────────────────────────────────────────────────────────────────
    print(f' Eval test {ty}...')
    # mean_ds/std_ds già disponibili dalla cache sopra
    ds_test=load_year(ty,snowy_only=USE_SNOWY_ONLY,time_stride=1)
    metrics=eval_year(ds_test,ckpt,mean_ds,std_ds)
    metrics.update(test_year=ty,val_year=vy,train_years=str(train_yrs))
    print(f' test={ty}: MAE={metrics["mae"]:.2f} BIAS={metrics["bias"]:+.2f} '
          f'RMSE={metrics["rmse"]:.2f} R²={metrics["r2"]:.3f} NSE={metrics["nse"]:.3f}')
    fold_results.append(metrics)
    del ds_test; gc.collect()

# ── Salva metriche ────────────────────────────────────────────────────────────
df_metrics=pd.DataFrame(fold_results)
cols=['test_year','val_year','n_pix','mae','bias','rmse','r2','nse']
df_metrics[cols].to_csv(FOLD_METRICS_CSV,index=False)
print(f'\nSalvato: {FOLD_METRICS_CSV}')
print(df_metrics[['test_year','mae','bias','rmse','r2','nse']].to_string(index=False))
print(f'\nMedia: MAE={df_metrics.mae.mean():.2f}  BIAS={df_metrics.bias.mean():+.2f}  '
      f'RMSE={df_metrics.rmse.mean():.2f}  R²={df_metrics.r2.mean():.3f}  '
      f'NSE={df_metrics.nse.mean():.3f}')


## Sezione 9 — XAI Permutation Importance

In [ ]:
# ── 16. XAI PERMUTATION IMPORTANCE ───────────────────────────────────────────
XAI_STRIDE   = 8
XAI_NPZ      = OUTPUT_DIR / 'xai_extended_importance.npz'
XAI_CSV      = OUTPUT_DIR / 'xai_extended_importance.csv'

CHANNEL_NAMES = INPUT_VARS + ['lat_norm','lon_norm'] + TOPO_VARS
CHANNEL_GROUP = (['temporal']*len(INPUT_VARS) + ['coord','coord'] +
                 ['topo']*len(TOPO_VARS))
assert len(CHANNEL_NAMES) == N_IN_CHANNELS


def eval_mae_zeroed(model, X_cpu, y_cpu, m_cpu, ch=None, batch_size=2):
    model.eval(); sa=0.0; sn=0
    T = X_cpu.shape[0]
    with torch.no_grad():
        for s in range(0,T,batch_size):
            Xb=X_cpu[s:s+batch_size].clone()
            if ch is not None: Xb[:,ch]=0.0
            pred=model(Xb.to(device)).cpu().numpy()[:,0]
            true=y_cpu[s:s+batch_size].numpy()[:,0]
            msk=m_cpu[s:s+batch_size].numpy()[:,0]>0
            pm=np.expm1(np.clip(pred,0,10)); tm=np.expm1(true)
            for b in range(pm.shape[0]):
                mk=msk[b]
                if mk.sum()==0: continue
                sa+=np.abs(pm[b][mk]-tm[b][mk]).sum(); sn+=mk.sum()
    return sa/sn if sn>0 else float('nan')


imp_matrix   = np.full((len(LOYO_FOLDS), N_IN_CHANNELS), np.nan, np.float32)
mae_baseline = np.full(len(LOYO_FOLDS), np.nan, np.float32)
xai_test_years = []

for fi, fold in enumerate(LOYO_FOLDS):
    ty=fold['test']; vy=fold['val']
    train_yrs=[y for y in YEARS_AVAILABLE if y not in (ty,vy)]
    ckpt=OUTPUT_DIR/CHECKPOINT_TEMPLATE.format(test=ty)
    if not ckpt.exists(): print(f'!! ckpt mancante test={ty}, skip'); continue

    print(f'\n{"="*60}\n XAI fold {fi+1}  test={ty}')
    xai_test_years.append(ty)

    stats_cache = OUTPUT_DIR / f"norm_stats_test{ty}.npz"
    if stats_cache.exists():
        sc = np.load(str(stats_cache), allow_pickle=True)
        mean_ds = xr.Dataset({v: xr.DataArray(float(sc["mean_arr"][i])) for i,v in enumerate(INPUT_VARS)})
        std_ds  = xr.Dataset({v: xr.DataArray(float(sc["std_arr"][i]))  for i,v in enumerate(INPUT_VARS)})
        print(f" Stats da cache ({stats_cache.name})")
    else:
        mean_ds, std_ds, _ = compute_norm_stats(train_yrs)
        np.savez(str(stats_cache),
                 mean_arr=np.array([float(mean_ds[v].values) for v in INPUT_VARS]),
                 std_arr =np.array([float(std_ds[v].values)  for v in INPUT_VARS]))
    ds_test=load_year(ty,snowy_only=USE_SNOWY_ONLY,time_stride=XAI_STRIDE)
    ds_eager=ds_test[INPUT_VARS+[TARGET_VAR,MASK_VAR]].load()
    dset=SWEZarrDataset(ds_eager,INPUT_VARS,TARGET_VAR,MASK_VAR,
                         mean_ds,std_ds,topo_tensor,augment=False,
                         add_doy=ADD_DOY,patch_size=None,lat_1d=lat_1d,lon_1d=lon_1d)
    Xs=[]; Ys=[]; Ms=[]
    for i in range(len(dset)): X,y,m=dset[i]; Xs.append(X); Ys.append(y); Ms.append(m)
    X_cpu=torch.stack(Xs); y_cpu=torch.stack(Ys); m_cpu=torch.stack(Ms)
    print(f'  cache: {tuple(X_cpu.shape)}  RAM≈{X_cpu.numel()*4/1e9:.2f} GB')

    model=UNet(N_IN_CHANNELS,dropout=DROPOUT_P).to(device)
    model.load_state_dict(torch.load(ckpt,map_location=device))

    mae_b=eval_mae_zeroed(model,X_cpu,y_cpu,m_cpu,ch=None)
    mae_baseline[fi]=mae_b; print(f'  baseline MAE={mae_b:.3f} mm')

    for c in range(N_IN_CHANNELS):
        mae_z=eval_mae_zeroed(model,X_cpu,y_cpu,m_cpu,ch=c)
        imp_matrix[fi,c]=mae_z-mae_b
        print(f'  ch{c:2d} {CHANNEL_NAMES[c]:<18s} imp={imp_matrix[fi,c]:+.2f} mm')

    del X_cpu,y_cpu,m_cpu,model,dset,ds_eager,ds_test; gc.collect()
    if device.type=='mps' and hasattr(torch.mps,'empty_cache'): torch.mps.empty_cache()
    elif device.type=='cuda': torch.cuda.empty_cache()

In [ ]:
# ── 17. SALVA E VISUALIZZA XAI ────────────────────────────────────────────────
np.savez(XAI_NPZ, imp_matrix=imp_matrix, fold_test_years=np.array(xai_test_years),
         mae_baseline=mae_baseline, channel_names=np.array(CHANNEL_NAMES),
         channel_group=np.array(CHANNEL_GROUP))

df_imp=pd.DataFrame(imp_matrix, columns=CHANNEL_NAMES,
                    index=[f'test_{y}' for y in xai_test_years])
df_imp.to_csv(XAI_CSV)
print(f'Salvato: {XAI_NPZ}\nSalvato: {XAI_CSV}')

imp_mean=np.nanmean(imp_matrix,axis=0)
imp_std =np.nanstd(imp_matrix,axis=0)
order   =np.argsort(-imp_mean)
mae_base_mean=np.nanmean(mae_baseline)

print(f'\nBaseline MAE (media): {mae_base_mean:.2f} mm')
print(f'{"Rank":>4}  {"Channel":<18s} {"Group":<10s}  {"mean imp":>10s}  {"std":>7s}  {"rel%":>7s}')
print('-'*70)
for r,i in enumerate(order,1):
    rel=100*imp_mean[i]/mae_base_mean if mae_base_mean>0 else 0
    print(f'  {r:>2d}  {CHANNEL_NAMES[i]:<18s} {CHANNEL_GROUP[i]:<10s}  '
          f'{imp_mean[i]:>+9.3f}   {imp_std[i]:>6.3f}  {rel:>+6.1f}%')

# ── Barplot ──────────────────────────────────────────────────────────────────
gc_color={'temporal':'#d32f2f','topo':'#388e3c','coord':'#616161'}
names_s=[CHANNEL_NAMES[i] for i in order]
grps_s =[CHANNEL_GROUP[i] for i in order]
mean_s =imp_mean[order]; std_s=imp_std[order]
colors =[gc_color.get(g,'grey') for g in grps_s]

fig,ax=plt.subplots(figsize=(10,8))
yp=np.arange(len(names_s))
ax.barh(yp,mean_s,xerr=std_s,color=colors,edgecolor='k',alpha=0.85,capsize=4)
ax.set_yticks(yp); ax.set_yticklabels(names_s); ax.invert_yaxis()
ax.axvline(0,color='k',lw=0.6)
ax.set_xlabel('MAE increase when channel zeroed [mm]')
ax.set_title(f'XAI permutation importance — Extended domain 1985–2024\n'
             f'baseline MAE={mae_base_mean:.2f} mm  (stride={XAI_STRIDE})')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=v,label=k) for k,v in gc_color.items()],loc='lower right')
ax.grid(axis='x',alpha=0.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'xai_barplot_extended.png',dpi=150); plt.show()

# ── Heatmap fold×channel ──────────────────────────────────────────────────────
fig,ax=plt.subplots(figsize=(14,max(4,len(xai_test_years)*0.5)))
vmax=max(np.abs(imp_matrix[:,order]).max(),0.5)
ax.imshow(imp_matrix[:,order],cmap='RdBu_r',vmin=-vmax,vmax=vmax,aspect='auto')
ax.set_xticks(range(N_IN_CHANNELS)); ax.set_xticklabels(names_s,rotation=45,ha='right')
ax.set_yticks(range(len(xai_test_years))); ax.set_yticklabels([str(y) for y in xai_test_years])
ax.set_title('Permutation importance per fold [mm MAE increase]')
plt.colorbar(ax.get_images()[0],ax=ax,label='imp [mm]')
for i in range(len(xai_test_years)):
    for j in range(N_IN_CHANNELS):
        v=imp_matrix[i,order[j]]
        if np.isfinite(v):
            ax.text(j,i,f'{v:+.1f}',ha='center',va='center',fontsize=7,
                    color='white' if abs(v)>0.5*vmax else 'black')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'xai_heatmap_extended.png',dpi=150); plt.show()

In [ ]:
# ── 18. CONFRONTO CON BASELINE (se disponibile) ───────────────────────────────
prev_npz=Path('sportlis_unet_NARR_lean_TPI_canopy_loyo_outputs/xai_loyo_lean_TPI_canopy_importance.npz')
if prev_npz.exists():
    prev=np.load(prev_npz,allow_pickle=True)
    prev_names=list(prev['channel_names'])
    prev_imp  =np.nanmean(prev['imp_matrix'],axis=0)
    prev_base =float(np.nanmean(prev['mae_baseline_per_fold']))
    print(f'Baseline lean+TPI+canopy (Idaho, 2017–2021): MAE={prev_base:.2f} mm')
    print(f'Extended model (dominio esteso, più anni):  MAE={mae_base_mean:.2f} mm')
    print(f'\n{"Channel":<18s} {"Extended":>12s}  {"Idaho-only":>12s}  {"delta":>10s}')
    print('-'*60)
    for name in CHANNEL_NAMES:
        ic=CHANNEL_NAMES.index(name); ve=imp_mean[ic]
        vp=float(prev_imp[prev_names.index(name)]) if name in prev_names else float('nan')
        d=ve-vp if np.isfinite(vp) else float('nan')
        print(f'{name:<18s} {ve:>+11.2f}  {vp:>+11.2f}  {d:>+9.2f}')
else:
    print(f'File baseline non trovato: {prev_npz}\nSkip confronto.')

print('\n' + '='*65)
print('  EXTENDED MODEL SUMMARY')
print('='*65)
print(f'  Dominio     : lat [{EXT_LAT_MIN},{EXT_LAT_MAX}]  lon [{EXT_LON_MIN},{EXT_LON_MAX}]')
print(f'  Griglia     : {EXT_H}×{EXT_W}  @ 3 km')
print(f'  Anni        : {YEARS_AVAILABLE[0]}–{YEARS_AVAILABLE[-1]}  ({len(YEARS_AVAILABLE)} anni)')
print(f'  LOYO_STEP   : {LOYO_STEP}  →  {len(LOYO_FOLDS)} fold')
print(f'  N_IN_CHAN   : {N_IN_CHANNELS}')
print(f'  Baseline MAE: {mae_base_mean:.2f} mm (XAI stride={XAI_STRIDE})')
if len(fold_results)>0:
    df=pd.DataFrame(fold_results)
    print(f'  Test MAE    : {df.mae.mean():.2f} ± {df.mae.std():.2f} mm')
    print(f'  Test R²     : {df.r2.mean():.3f} ± {df.r2.std():.3f}')
    print(f'  Test NSE    : {df.nse.mean():.3f} ± {df.nse.std():.3f}')
print('='*65)